# ✨ IntelliCode-SL | Formatter SLM — fp16 Full Precision

**Model:** `meta-llama/Llama-3.2-1B-Instruct`
**Config:** fp16 base + full precision adapter (via Unsloth)

Run this first → then **09B** (4-bit) → then **09C** (8-bit) each in a fresh session.

> T4 GPU required. Run cells top to bottom.

In [ ]:
# ── Cell 1: Install ────────────────────────────────────────────
!pip install -q unsloth transformers peft accelerate bitsandbytes
print("✅ Dependencies installed")

In [ ]:
# ── Cell 2: Imports ────────────────────────────────────────────
import os, torch, time, gc, json
from google.colab import drive
from huggingface_hub import login
from unsloth import FastLanguageModel
from peft import PeftModel

os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'

print("✅ Imports done")
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND")
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")


In [ ]:
# ── Cell 3: Config + Drive + Login ─────────────────────────────
drive.mount("/content/drive")

HF_TOKEN     = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXX"  # ← paste your token
ADAPTER_PATH = "/content/drive/MyDrive/IntelliCode-SL/adapters/formatter_adapter"
MODEL_NAME   = "meta-llama/Llama-3.2-1B-Instruct"
MAX_SEQ_LEN  = 2048
TASKS        = ["debug", "generate", "explain", "modify"]

login(token=HF_TOKEN)
print("✅ Drive mounted + HF login done")
print(f"   Adapter: {ADAPTER_PATH}")


In [ ]:
# ── Cell 4: Prompt Template ────────────────────────────────────
PROMPT_TEMPLATE = """### Task:
You are a response formatter. Take the raw agent output below and rewrite it as a clean, professional, well-structured response that is easy for the user to read and understand.
Do not add new information. Only improve the clarity, structure, and readability of the existing content.

### Raw Agent Output:
{input}

### Formatted Response:
"""

print("✅ Prompt template defined")


In [ ]:
# ── Cell 5: Test Cases (25 per agent type = 100 total) ─────────
test_cases = {
    "debug": [
        {"input": "- bug found on line 8: used + instead of -\n- missing null check at start\n- fixed both issues\n- function now returns correct result", "expected_contains": "line 8"},
        {"input": "- off by one error in loop\n- range should start at 0 not 1\n- corrected the range call\n- all test cases now pass", "expected_contains": "loop"},
        {"input": "- division by zero error when denominator is 0\n- added check before division\n- returns None if denominator is 0\n- added unit test for edge case", "expected_contains": "zero"},
        {"input": "- wrong operator: used == instead of =\n- variable was never being assigned\n- fixed assignment\n- function behaves correctly now", "expected_contains": "operator"},
        {"input": "- missing return statement in else branch\n- function returned None unexpectedly\n- added return False to else\n- logic is now complete", "expected_contains": "return"},
        {"input": "- key error when dict key does not exist\n- replaced direct access with .get()\n- added default value of 0\n- no more crashes on missing keys", "expected_contains": "key"},
        {"input": "- infinite loop detected: counter not incrementing\n- added i += 1 inside while loop\n- loop now terminates correctly\n- performance is normal", "expected_contains": "loop"},
        {"input": "- type error: string passed where int expected\n- added int() conversion at input\n- validation added for non-numeric input\n- function now handles bad input gracefully", "expected_contains": "type"},
        {"input": "- index out of range on last element\n- len(lst) should be len(lst)-1\n- fixed the boundary condition\n- list access is now safe", "expected_contains": "index"},
        {"input": "- mutable default argument bug\n- list=[] in function signature shared across calls\n- changed to list=None and initialized inside\n- each call now gets a fresh list", "expected_contains": "default"},
        {"input": "- scope error: variable used before assignment\n- moved variable initialization outside if block\n- function now has correct variable scope\n- no more UnboundLocalError", "expected_contains": "scope"},
        {"input": "- recursion missing base case\n- function called itself infinitely\n- added base case for n == 0\n- recursion now terminates", "expected_contains": "base case"},
        {"input": "- wrong comparison: using 'is' instead of '=='\n- identity check was failing for equal strings\n- replaced 'is' with '=='\n- comparison now works correctly", "expected_contains": "comparison"},
        {"input": "- file not closed after reading\n- resource leak detected\n- refactored to use 'with open()' context manager\n- file is now always properly closed", "expected_contains": "file"},
        {"input": "- list modified during iteration causing skip\n- created copy of list before iterating\n- all elements now processed correctly\n- no more elements being skipped", "expected_contains": "iteration"},
        {"input": "- global variable modified in function without global keyword\n- counter was not updating globally\n- added global counter declaration\n- global state now updates correctly", "expected_contains": "global"},
        {"input": "- regex pattern incorrect: missing escape for dot\n- '.' was matching any character\n- replaced with '\\.' to match literal dot\n- email validation now works correctly", "expected_contains": "pattern"},
        {"input": "- async function called without await\n- coroutine object returned instead of result\n- added await keyword before call\n- async function now executes correctly", "expected_contains": "await"},
        {"input": "- exception too broad: bare except clause\n- catching all exceptions including KeyboardInterrupt\n- narrowed to except ValueError\n- only expected errors are now caught", "expected_contains": "exception"},
        {"input": "- float precision error in comparison\n- 0.1 + 0.2 != 0.3 due to floating point\n- replaced with math.isclose()\n- comparison now handles float precision correctly", "expected_contains": "float"},
        {"input": "- class method missing self parameter\n- method treated as static function\n- added self as first parameter\n- instance method works correctly now", "expected_contains": "self"},
        {"input": "- list comprehension filtering wrong items\n- condition was inverted\n- fixed condition from != to ==\n- only correct items are now filtered", "expected_contains": "condition"},
        {"input": "- string concatenation in loop is O(n^2)\n- rebuilt string by joining list instead\n- performance improved from O(n^2) to O(n)\n- function now scales properly", "expected_contains": "performance"},
        {"input": "- dictionary keys iterated while modifying\n- RuntimeError raised during iteration\n- iterated over list(dict.keys()) instead\n- no more runtime errors", "expected_contains": "dictionary"},
        {"input": "- timezone-naive datetime compared with timezone-aware\n- TypeError raised on comparison\n- added UTC timezone to naive datetime\n- datetime comparison now works correctly", "expected_contains": "datetime"},
    ],
    "generate": [
        {"input": "- wrote function is_prime(n)\n- handles edge cases: n < 2 returns False\n- checks divisibility up to sqrt(n)\n- returns True if no divisors found", "expected_contains": "prime"},
        {"input": "- implemented binary search\n- takes sorted array and target value\n- uses left/right pointers with midpoint\n- returns index or -1 if not found", "expected_contains": "binary search"},
        {"input": "- created Stack class\n- push() appends to internal list\n- pop() removes and returns last item\n- peek() returns last without removing\n- is_empty() checks if list is empty", "expected_contains": "Stack"},
        {"input": "- wrote merge sort implementation\n- recursively splits array at midpoint\n- merges sorted halves back together\n- time complexity is O(n log n)", "expected_contains": "merge sort"},
        {"input": "- implemented LRU cache\n- used OrderedDict to track access order\n- get() moves key to end on access\n- put() evicts oldest if over capacity", "expected_contains": "LRU"},
        {"input": "- created fibonacci generator\n- uses yield for memory efficiency\n- maintains previous two values only\n- can generate infinite sequence", "expected_contains": "fibonacci"},
        {"input": "- wrote decorator for timing functions\n- wraps function with time.time() calls\n- prints elapsed time after execution\n- returns original function result", "expected_contains": "decorator"},
        {"input": "- implemented quicksort algorithm\n- picks pivot as middle element\n- partitions into less, equal, greater\n- recursively sorts partitions", "expected_contains": "quicksort"},
        {"input": "- wrote word frequency counter\n- splits text into words\n- uses dict to count occurrences\n- returns sorted by frequency descending", "expected_contains": "frequency"},
        {"input": "- implemented graph BFS\n- uses deque for efficient queue operations\n- tracks visited nodes to avoid cycles\n- returns nodes in traversal order", "expected_contains": "BFS"},
        {"input": "- created Caesar cipher encoder\n- shifts each letter by given amount\n- handles uppercase and lowercase\n- non-alpha characters unchanged", "expected_contains": "cipher"},
        {"input": "- wrote palindrome checker\n- strips whitespace and lowercases\n- compares string to its reverse\n- returns True if palindrome", "expected_contains": "palindrome"},
        {"input": "- implemented flatten function\n- handles arbitrarily nested lists\n- uses recursion with isinstance check\n- returns single flat list", "expected_contains": "flatten"},
        {"input": "- created memoize decorator\n- uses dict to cache results by args\n- checks cache before computing\n- returns cached result if available", "expected_contains": "memoize"},
        {"input": "- wrote rate limiter class\n- tracks request timestamps in deque\n- rejects requests over limit per window\n- thread-safe with lock", "expected_contains": "rate limit"},
        {"input": "- implemented trie data structure\n- insert() adds word character by character\n- search() traverses trie for word\n- end marker '#' signals complete word", "expected_contains": "trie"},
        {"input": "- wrote context manager for DB connection\n- __enter__ opens connection\n- __exit__ closes connection always\n- handles exceptions gracefully", "expected_contains": "context manager"},
        {"input": "- implemented observer pattern\n- Subject class holds list of observers\n- notify() calls update() on all observers\n- observers can subscribe/unsubscribe", "expected_contains": "observer"},
        {"input": "- created retry decorator\n- retries function up to n times on exception\n- uses exponential backoff between retries\n- raises last exception if all retries fail", "expected_contains": "retry"},
        {"input": "- wrote run-length encoder\n- iterates through string counting repeats\n- appends count+char to result list\n- joins result into encoded string", "expected_contains": "run-length"},
        {"input": "- implemented sliding window max\n- uses monotonic deque\n- removes indices outside window\n- removes smaller elements from back", "expected_contains": "sliding window"},
        {"input": "- wrote topological sort\n- builds adjacency list from edges\n- uses DFS with visited set\n- appends nodes to stack post-DFS", "expected_contains": "topological"},
        {"input": "- created singleton class\n- uses class variable to store instance\n- __new__ checks if instance exists\n- thread-safe with lock", "expected_contains": "singleton"},
        {"input": "- implemented matrix multiplication\n- validates dimensions are compatible\n- uses three nested loops\n- returns result matrix", "expected_contains": "matrix"},
        {"input": "- wrote Levenshtein distance function\n- builds 2D DP table\n- fills in edit costs row by row\n- returns bottom-right cell as answer", "expected_contains": "Levenshtein"},
    ],
    "explain": [
        {"input": "- function takes list as input\n- iterates through each element\n- multiplies each by 2\n- returns new list with doubled values", "expected_contains": "doubles"},
        {"input": "- class stores key-value pairs\n- hash function maps keys to buckets\n- handles collisions with chaining\n- supports get, set, delete operations", "expected_contains": "hash"},
        {"input": "- function checks if number is even\n- uses modulo operator %\n- returns True if remainder is 0\n- returns False otherwise", "expected_contains": "even"},
        {"input": "- recursively computes factorial\n- base case: n == 0 returns 1\n- recursive case: n * factorial(n-1)\n- call stack grows with each call", "expected_contains": "factorial"},
        {"input": "- decorator wraps original function\n- adds timing logic before and after\n- measures elapsed time with time.time()\n- prints result without changing return value", "expected_contains": "timing"},
        {"input": "- generator yields values one at a time\n- uses yield keyword instead of return\n- caller gets values lazily\n- memory efficient for large sequences", "expected_contains": "generator"},
        {"input": "- context manager manages resources\n- __enter__ acquires resource\n- __exit__ always releases resource\n- used with 'with' statement", "expected_contains": "resource"},
        {"input": "- binary search works on sorted arrays\n- checks midpoint element each iteration\n- eliminates half the array each step\n- O(log n) time complexity", "expected_contains": "binary search"},
        {"input": "- quicksort picks a pivot element\n- partitions array around pivot\n- recursively sorts each partition\n- average O(n log n) complexity", "expected_contains": "pivot"},
        {"input": "- class implements iterator protocol\n- __iter__ returns self\n- __next__ returns next value\n- raises StopIteration when done", "expected_contains": "iterator"},
        {"input": "- function uses dynamic programming\n- stores subproblem results in table\n- builds up solution bottom-up\n- avoids redundant computations", "expected_contains": "dynamic programming"},
        {"input": "- regex pattern matches email addresses\n- checks for local part before @\n- validates domain and TLD format\n- returns True if pattern matches", "expected_contains": "email"},
        {"input": "- async function runs without blocking\n- uses await to pause at I/O\n- allows other coroutines to run\n- event loop manages execution", "expected_contains": "async"},
        {"input": "- linked list stores nodes with pointers\n- each node holds value and next reference\n- traversal starts from head node\n- insertion at head is O(1)", "expected_contains": "linked list"},
        {"input": "- heap maintains min or max at root\n- push adds element and bubbles up\n- pop removes root and heapifies down\n- used for priority queue implementation", "expected_contains": "heap"},
        {"input": "- function implements memoization\n- caches results in dictionary\n- checks cache before computing\n- drastically reduces repeated computations", "expected_contains": "cache"},
        {"input": "- lambda creates anonymous function\n- takes arguments before colon\n- expression after colon is returned\n- often used with map, filter, sorted", "expected_contains": "lambda"},
        {"input": "- class uses __slots__ instead of __dict__\n- reduces memory per instance\n- only listed attributes allowed\n- faster attribute access", "expected_contains": "memory"},
        {"input": "- function uses list comprehension\n- shorter and faster than for loop\n- filters and transforms in one line\n- returns new list with results", "expected_contains": "comprehension"},
        {"input": "- GIL prevents true parallel threads\n- only one thread runs Python at a time\n- use multiprocessing for CPU-bound work\n- asyncio works around GIL for I/O", "expected_contains": "GIL"},
        {"input": "- property decorator creates getter\n- @x.setter creates setter method\n- allows validation on assignment\n- hides internal implementation", "expected_contains": "property"},
        {"input": "- function uses walrus operator :=\n- assigns and evaluates in one step\n- useful in while loop conditions\n- reduces lines of code", "expected_contains": "walrus"},
        {"input": "- dataclass auto-generates __init__\n- field types declared as annotations\n- also generates __repr__ and __eq__\n- cleaner than manual class definition", "expected_contains": "dataclass"},
        {"input": "- function raises custom exception\n- inherits from base Exception class\n- adds domain-specific error message\n- caller can catch specifically", "expected_contains": "exception"},
        {"input": "- function uses zip to pair two lists\n- iterates both lists simultaneously\n- stops at shortest list\n- returns tuples of paired elements", "expected_contains": "zip"},
    ],
    "modify": [
        {"input": "- added type hints to all parameters\n- added docstring explaining purpose\n- added input validation for negative numbers\n- refactored loop to list comprehension\n- added logging for debug purposes", "expected_contains": "type hints"},
        {"input": "- wrapped function in try/except block\n- catches ValueError and TypeError\n- logs error before re-raising\n- returns default on recoverable errors", "expected_contains": "error handling"},
        {"input": "- added @lru_cache decorator\n- cache size set to 128\n- recursive calls now reuse results\n- significant speedup on repeated inputs", "expected_contains": "cache"},
        {"input": "- converted sync function to async\n- added async keyword to def\n- added await before I/O calls\n- caller must now use await", "expected_contains": "async"},
        {"input": "- added pagination support\n- accepts page and page_size params\n- slices results accordingly\n- returns total count with results", "expected_contains": "pagination"},
        {"input": "- replaced if/elif chain with dict lookup\n- cleaner and more extensible\n- default value handles unknown keys\n- O(1) lookup instead of O(n)", "expected_contains": "dictionary"},
        {"input": "- added @property decorator to getter\n- added setter with validation\n- private attribute now _value\n- encapsulation improved", "expected_contains": "property"},
        {"input": "- added __enter__ and __exit__ methods\n- resource acquired in __enter__\n- resource released in __exit__\n- class now usable as context manager", "expected_contains": "context manager"},
        {"input": "- added threading.Lock to class\n- lock acquired before state change\n- lock released after state change\n- class is now thread-safe", "expected_contains": "thread-safe"},
        {"input": "- added retry logic with backoff\n- retries up to 3 times on failure\n- waits 2^n seconds between attempts\n- raises exception after all retries", "expected_contains": "retry"},
        {"input": "- refactored to use generator\n- replaced return list with yield\n- memory usage reduced significantly\n- caller iterates lazily", "expected_contains": "generator"},
        {"input": "- added observer list to class\n- subscribe() adds observer\n- unsubscribe() removes observer\n- notify() calls update on all observers", "expected_contains": "observer"},
        {"input": "- added os.getenv calls for config\n- hardcoded values replaced with env vars\n- added defaults for missing env vars\n- config now injectable at runtime", "expected_contains": "environment"},
        {"input": "- added to_dict() method\n- added from_dict() class method\n- all fields included in serialization\n- supports JSON export", "expected_contains": "serialization"},
        {"input": "- added audit log on create and delete\n- logs user, timestamp, and action\n- stored in separate audit table\n- queryable for compliance", "expected_contains": "audit"},
        {"input": "- added health check endpoint\n- checks DB connection\n- checks cache connection\n- returns status dict with each component", "expected_contains": "health"},
        {"input": "- added role parameter to method\n- checks role before executing\n- raises PermissionError if unauthorized\n- admin role bypasses all checks", "expected_contains": "role"},
        {"input": "- added gzip compression option\n- compresses output if flag is True\n- uses gzip.open instead of open\n- file size reduced significantly", "expected_contains": "compression"},
        {"input": "- added timeout parameter to request\n- default timeout set to 30 seconds\n- raises TimeoutError on expiry\n- prevents hanging requests", "expected_contains": "timeout"},
        {"input": "- added feature flag check\n- reads flag from config\n- new code path only runs if flag enabled\n- old path preserved as fallback", "expected_contains": "feature flag"},
        {"input": "- added signal handler for SIGTERM\n- gracefully stops accepting requests\n- waits for in-flight requests to complete\n- logs shutdown sequence", "expected_contains": "shutdown"},
        {"input": "- replaced f-string with locale.format\n- supports multiple date formats\n- detects locale from user settings\n- falls back to ISO format", "expected_contains": "locale"},
        {"input": "- added parameterized queries\n- replaced string formatting with ?\n- prevents SQL injection attacks\n- values passed as tuple to execute", "expected_contains": "SQL injection"},
        {"input": "- added rate limit decorator\n- tracks calls per time window\n- raises RateLimitError when exceeded\n- configurable limit and window", "expected_contains": "rate limit"},
        {"input": "- added lazy loading flag\n- data loaded only on first access\n- cached after first load\n- reduces startup time", "expected_contains": "lazy"},
    ],
}

print(f"✅ Test cases ready")
for task, cases in test_cases.items():
    print(f"   {task:<10} : {len(cases)} cases")
print(f"   {'TOTAL':<10} : {sum(len(v) for v in test_cases.values())} cases")


In [ ]:
# ── Cell 6: Evaluation Helper ──────────────────────────────────
def run_inference(model, tokenizer, input_text, max_new_tokens=250):
    prompt = PROMPT_TEMPLATE.format(input=input_text.strip())
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LEN
    ).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("### Formatted Response:")[-1].strip()

def evaluate(model, tokenizer, config_name):
    print(f"\n{'='*58}")
    print(f"  Evaluating: {config_name}")
    print(f"{'='*58}")

    all_results = []
    per_task = {task: {"pass": 0, "fail": 0, "total": 0, "times": []} for task in TASKS}
    total_time = 0
    idx = 0

    for task, cases in test_cases.items():
        print(f"\n  Running {task} ({len(cases)} cases)...")
        for case in cases:
            t0 = time.time()
            output = run_inference(model, tokenizer, case["input"])
            elapsed = time.time() - t0
            total_time += elapsed
            per_task[task]["times"].append(elapsed)
            per_task[task]["total"] += 1

            expected = case.get("expected_contains", "")
            passed = (
                len(output) > 30
                and not output.strip().startswith("###")
                and expected.lower() in output.lower()
            )
            if passed:
                per_task[task]["pass"] += 1
            else:
                per_task[task]["fail"] += 1

            all_results.append({
                "task": task,
                "input": case["input"][:80],
                "expected": expected,
                "output": output[:300],
                "passed": passed,
                "time_ms": elapsed * 1000
            })
            idx += 1
            if idx % 25 == 0:
                total_pass = sum(v["pass"] for v in per_task.values())
                print(f"  [{idx:>3}/100]  running pass rate: {total_pass/idx*100:.1f}%")

    total_pass = sum(v["pass"] for v in per_task.values())
    total_cases = sum(v["total"] for v in per_task.values())
    avg_ms = total_time / total_cases * 1000

    print(f"\n  ✅ Overall Pass Rate : {total_pass/total_cases*100:.2f}% ({total_pass}/{total_cases})")
    print(f"  ⏱  Avg latency       : {avg_ms:.1f} ms/prompt")
    print(f"\n  {'Task':<12} {'Pass':>6} {'Fail':>6} {'Total':>7} {'Rate':>8} {'Avg ms':>8}")
    print(f"  {'-'*50}")
    for task in TASKS:
        s = per_task[task]
        rate = s["pass"] / s["total"] * 100
        avg_t = sum(s["times"]) / len(s["times"]) * 1000
        print(f"  {task:<12} {s['pass']:>6} {s['fail']:>6} {s['total']:>7} {rate:>7.1f}% {avg_t:>7.1f}ms")

    return {
        "config": config_name,
        "pass_rate": total_pass / total_cases * 100,
        "total_pass": total_pass,
        "total": total_cases,
        "avg_time_ms": avg_ms,
        "per_task": {
            task: {
                "pass": per_task[task]["pass"],
                "fail": per_task[task]["fail"],
                "total": per_task[task]["total"],
                "pass_rate": per_task[task]["pass"] / per_task[task]["total"] * 100,
                "avg_time_ms": sum(per_task[task]["times"]) / len(per_task[task]["times"]) * 1000
            } for task in TASKS
        },
        "samples": all_results
    }

print("✅ Evaluation helper defined")


In [ ]:
# ── Cell 7: Load Model — fp16 via Unsloth ──────────────────────
print("Loading fp16 full precision model via Unsloth...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = torch.float16,
    load_in_4bit   = False,
    token          = HF_TOKEN,
)
model = PeftModel.from_pretrained(model, ADAPTER_PATH, torch_dtype=torch.float16)
FastLanguageModel.for_inference(model)
model.eval()

print("✅ Model loaded")
print(f"   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")


In [ ]:
# ── Cell 8: Run Evaluation ─────────────────────────────────────
results = evaluate(model, tokenizer, "Formatter SLM — fp16 full precision")

In [ ]:
# ── Cell 9: Sample Output Inspection ──────────────────────────
print("Sample outputs (first 2 per task):\n")
shown = {t: 0 for t in TASKS}
for r in results["samples"]:
    task = r["task"]
    if shown[task] < 2:
        print(f"  [{task.upper()}]")
        print(f"  Input    : {r['input'][:70]}...")
        print(f"  Expected : {r['expected']}")
        print(f"  Output   : {r['output'][:200]}...")
        print(f"  Passed   : {'✅' if r['passed'] else '❌'} | {r['time_ms']:.0f}ms")
        print()
        shown[task] += 1


In [ ]:
# ── Cell 10: Save Results + Unload ─────────────────────────────
save_path = "/content/drive/MyDrive/IntelliCode-SL/benchmarks/formatter_slm_fp16.json"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
with open(save_path, "w") as f:
    json.dump(results, f, indent=2)

assert os.path.exists(save_path), "❌ Save failed"
print(f"✅ Results saved → {save_path}")
print(f"   File size: {os.path.getsize(save_path)/1024:.1f} KB")

del model, tokenizer
gc.collect()
torch.cuda.empty_cache()
print(f"✅ Unloaded | VRAM now: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print("\n  Run 09B_Formatter_4bit.ipynb next in a fresh session.")
